# Model Consumption: Aliases, SQL Predict, and Cross-Language Models

This notebook focuses on **consuming** models from the Snowflake Model Registry
using R -- regardless of whether they were trained in R or Python.

A common workflow in data teams: a **Python data scientist** trains and registers
a model, and an **R analyst** consumes it for scoring, reporting, or monitoring.
Snowflake Workspace Notebooks make this seamless -- both languages work in the
same notebook, and the Model Registry is the shared artifact store.

**What you'll learn:**
- Managing model aliases for production workflows
- Querying model metadata with SQL-direct functions
- How a Python DS trains and registers a model (pure Python cell)
- Training the same model from R via `reticulate` (bilingual approach)
- Running SQL batch inference (`sfr_predict_sql`) against warehouse-targeted models
- Comparing R and Python model predictions side-by-side
- Online inference via SPCS for low-latency use cases
- Continuous inference with Dynamic Tables and inference Feature Views

**Prerequisites:** Complete the Model Registry notebook first to understand
basic model registration. This notebook builds on those concepts.

**Sections:**
1. Setup
2. Connect & Version Check
3. Register an R Model
4. Model Aliases
5. Model Info (SQL-direct)
6. Cross-Language Model Training
7. SQL-direct Batch Inference
8. Side-by-Side Comparison
9. Container Inference (SPCS)
10. Continuous Inference: Dynamic Tables & Inference Feature Views
11. Cleanup


In [ ]:
import time as _time
_nb_start = _time.time()
print(f"Notebook started: {_time.strftime('%Y-%m-%d %H:%M:%S')}")


## 1. Setup

Single cell to set session context, validate EAI, install R, and install packages.


In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_config.yaml", packages=["snowflakeR"])


## 2. Connect & Version Check


In [ ]:
%%R
library(snowflakeR)

conn <- sfr_connect()
conn

reg <- sfr_model_registry(conn)

# Verify snowflake-ml-python version
ml_ver <- sfr_ml_version()
rcat(sprintf("snowflake-ml-python version: %s", ml_ver))


---

## 3. Register an R Model

First, we register a simple R linear model so we can demonstrate aliases and
compare it with a Python model later.


In [ ]:
%%R
# Train on mtcars (same dataset we'll use for the Python model)
model_r <- lm(mpg ~ wt + hp, data = mtcars)

mv_r <- sfr_log_model(
  reg,
  model        = model_r,
  model_name   = "SFR_DEMO_CONSUMPTION",
  version_name = "R_V1",
  input_cols   = list(wt = "double", hp = "double"),
  output_cols  = list(prediction = "double"),
  comment      = "R lm: MPG from weight + horsepower",
  task         = "TABULAR_REGRESSION"
)

mv_r


---

## 4. Model Aliases

Aliases are human-readable labels you attach to specific model versions.
Unlike the **default version** (which is a single pointer), you can have
multiple aliases on the same model -- e.g., `production`, `staging`, `champion`.

Aliases are SQL identifiers (unquoted), not string literals.


In [ ]:
%%R
# Set an alias on the R model version
sfr_set_model_alias(conn, "SFR_DEMO_CONSUMPTION", "R_V1", "champion")
rcat("Alias 'champion' set on R_V1")

# You can use aliases in SQL queries:
# SELECT MODEL(SFR_DEMO_CONSUMPTION, champion)!predict(...) FROM ...


In [ ]:
%%R
# Remove the alias
sfr_unset_model_alias(conn, "SFR_DEMO_CONSUMPTION", "champion")
rcat("Alias 'champion' removed")


---

## 5. Model Info (SQL-direct)

`sfr_model_info()` queries `INFORMATION_SCHEMA.MODEL_VERSIONS` to retrieve
metadata about registered models without loading the Python SDK. Useful for
auditing, compliance, and quick version comparisons.


In [ ]:
%%R
# Get info for a specific model
info <- sfr_model_info(conn, model_name = "SFR_DEMO_CONSUMPTION")
info


---

## 6. Cross-Language Model Training

The Model Registry is language-agnostic. A **Python data scientist** can train
and register a model, and an **R analyst** can consume it -- or vice versa.

Snowflake Workspace Notebooks support both Python and R cells in the same
notebook (`%%R` magic for R cells), so bilingual teams can collaborate directly.

We show two approaches for the same sklearn model:
- **6a.** Pure Python (how a DS would typically do it)
- **6b.** From R via `reticulate` (bilingual approach)

Both produce a warehouse-targeted model that supports SQL-native batch inference
via `MODEL()!predict()`. R models, by contrast, target `SNOWPARK_CONTAINER_SERVICES`
because they need `rpy2` + `r-base` at inference time.


### 6a. Pure Python approach (how a data scientist would do it)

This is what the Python cell looks like when a DS trains and registers the
model. In a Workspace Notebook this would be a regular Python cell (no `%%R`
magic). The R analyst in the team doesn't need to touch this cell -- they
just consume the model from Section 7 onwards.

**Note:** This cell is shown as reference markdown. If you're running this
notebook in bilingual mode, you could paste this into a Python cell instead.

```python
# Python cell -- no %%R magic
from snowflake.ml.registry import Registry
from snowflake.ml.model.task import Task
from sklearn.linear_model import LinearRegression
import pandas as pd

# Load the mtcars dataset (available in Snowflake sample data or uploaded)
df = session.table('MTCARS').to_pandas()

# Train sklearn LinearRegression
X = df[['WT', 'HP']].values
y = df['MPG'].values
model = LinearRegression().fit(X, y)

print(f'Coefficients: wt={model.coef_[0]:.4f}, hp={model.coef_[1]:.4f}')
print(f'Intercept:    {model.intercept_:.4f}')

# Register to Model Registry with WAREHOUSE targeting
reg = Registry(session, database_name=DB, schema_name=SCHEMA)

mv = reg.log_model(
    model=model,
    model_name='SFR_DEMO_CONSUMPTION',
    version_name='PY_V1',
    sample_input_data=pd.DataFrame({'wt': [2.5], 'hp': [110.0]}),
    target_platforms=['WAREHOUSE'],
    task=Task.TABULAR_REGRESSION,
    comment='Python sklearn LinearRegression: MPG from wt + hp',
)
print(f'Registered: {mv.model_name}/{mv.version_name}')
```

Once registered, the model is available to **any** language -- R, Python, SQL --
via `MODEL()!predict()` or the Registry API.


### 6b. From R via `reticulate` (bilingual approach)

If you prefer to stay in R, `reticulate` lets you call sklearn directly.
This is useful when R users want to train Python models without switching
cells, or when the training data is already in R objects.


In [ ]:
%%R
# Prepare the data as R objects
train_X <- mtcars[, c("wt", "hp")]
train_y <- mtcars$mpg

rcat(sprintf("Training data: %d rows, %d features", nrow(train_X), ncol(train_X)))


In [ ]:
%%R
# Import sklearn via reticulate and train a LinearRegression
sklearn_lm <- reticulate::import("sklearn.linear_model")

py_model <- sklearn_lm$LinearRegression()
py_model$fit(as.matrix(train_X), train_y)

# Show coefficients (should match R's lm closely)
coefs <- py_model$coef_
names(coefs) <- c("wt", "hp")
rcat(sprintf("Python sklearn coefficients: wt=%.4f, hp=%.4f", coefs[1], coefs[2]))
rcat(sprintf("Python sklearn intercept:    %.4f", py_model$intercept_))

# Compare with R
rcat(sprintf("R lm() coefficients:         wt=%.4f, hp=%.4f",
             coef(model_r)["wt"], coef(model_r)["hp"]))
rcat(sprintf("R lm() intercept:            %.4f", coef(model_r)["(Intercept)"]))


### Register the Python model to the warehouse

We use `reticulate` to call the Python `Registry.log_model()` directly.
The key difference from `sfr_log_model()` is `target_platforms=["WAREHOUSE"]`
which enables SQL-native inference without needing SPCS.


In [ ]:
%%R
# Register the sklearn model via the Python Registry API
py_registry <- reticulate::import("snowflake.ml.registry")
py_task     <- reticulate::import("snowflake.ml.model.task")

py_reg <- py_registry$Registry(
  session       = conn$session,
  database_name = conn$database,
  schema_name   = conn$schema
)

# Create sample input for signature inference
pd <- reticulate::import("pandas")
sample_df <- pd$DataFrame(list(
  wt = list(2.5),
  hp = list(110.0)
))

py_mv <- py_reg$log_model(
  model             = py_model,
  model_name        = "SFR_DEMO_CONSUMPTION",
  version_name      = "PY_V1",
  sample_input_data = sample_df,
  target_platforms  = list("WAREHOUSE"),
  task              = py_task$Task$TABULAR_REGRESSION,
  comment           = "Python sklearn LinearRegression: MPG from wt + hp"
)

rcat(sprintf("Python model registered: %s version %s",
             py_mv$model_name, py_mv$version_name))
rcat("Target platform: WAREHOUSE (SQL-native inference enabled)")


---

## 7. SQL-direct Batch Inference (`sfr_predict_sql`)

`sfr_predict_sql()` generates and executes a `MODEL()!predict()` SQL query.
This runs in the warehouse -- no SPCS container needed, no client-side Python.

**How it works under the hood:**

When you call `log_model()`, the SDK serialises the model, generates Python
wrapper code, and uploads everything to an internal stage. It then runs
`CREATE MODEL ... FROM @stage` to register it as a first-class schema-level
object. The predict function is a Python function stored with the model --
you can inspect it via `SHOW FUNCTIONS IN MODEL <model> VERSION <version>`
(the `language` column shows `PYTHON`). At query time the warehouse executes
this Python function using the Python runtime pre-installed in the warehouse
image (from Snowflake's Anaconda channel).

It works for any model logged with `target_platforms = ["WAREHOUSE"]`.

The generated SQL looks like:
```sql
WITH m AS MODEL SFR_DEMO_CONSUMPTION VERSION PY_V1
SELECT m!predict(*) AS prediction
FROM (<your_sql>)
```


In [ ]:
%%R
# Create a scoring table
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", sfr_fqn(conn, "SFR_DEMO_SCORING"), "(",
  "  wt FLOAT, hp FLOAT",
  ")"
))

sfr_execute(conn, paste(
  "INSERT INTO", sfr_fqn(conn, "SFR_DEMO_SCORING"), "VALUES",
  "(2.620, 110), (3.440, 175), (3.570, 245), (2.200, 93)"
))

rcat("Scoring table created with 4 rows.")


In [ ]:
%%R
# Run batch inference via SQL against the Python sklearn model
preds <- sfr_predict_sql(
  conn,
  model_name   = "SFR_DEMO_CONSUMPTION",
  sql          = paste("SELECT wt, hp FROM", sfr_fqn(conn, "SFR_DEMO_SCORING")),
  version_name = "PY_V1"
)

rcat("SQL-native predictions (warehouse, no SPCS):")
preds


The result is a plain R `data.frame` -- ready for downstream analysis in R.

**Key point:** This works because the sklearn model targets `WAREHOUSE` --
its dependencies (`scikit-learn`, `numpy`) are available in the warehouse's
Anaconda channel. The model's Python predict function runs inside the warehouse
using the pre-installed Python runtime.

R models target `SNOWPARK_CONTAINER_SERVICES` because `rpy2` + `r-base` are
not in the warehouse Anaconda channel. For R models, use `sfr_predict_local()`
or deploy via SPCS.


---

## 8. Side-by-Side Comparison

Let's compare predictions from the R `lm()` model (local) with the
Python sklearn model (SQL warehouse inference) on the same data.


In [ ]:
%%R
# R model: local prediction
scoring_data <- data.frame(wt = c(2.620, 3.440, 3.570, 2.200),
                           hp = c(110, 175, 245, 93))

r_preds <- predict(model_r, scoring_data)

# sfr_predict_sql returns input columns + parsed prediction column.
# Merge on inputs to align rows (SQL doesn't guarantee order).
scoring_data$R_lm <- round(r_preds, 4)
preds$Py_sklearn  <- round(preds$output_feature_0, 4)

comparison <- merge(scoring_data, preds[, c("WT", "HP", "Py_sklearn")],
                    by.x = c("wt", "hp"), by.y = c("WT", "HP"))
comparison$diff <- round(comparison$R_lm - comparison$Py_sklearn, 6)

rcat("R lm() vs Python sklearn predictions:")
comparison


The predictions should be nearly identical -- both are OLS linear regression
on the same data. Minor floating-point differences (< 1e-10) are expected.

**Takeaway:** R users can consume *any* model in the registry. For Python models
logged with `WAREHOUSE` targeting, `sfr_predict_sql()` provides a SQL
inference path that's fast, scalable, and requires no client-side Python.
The warehouse handles the Python execution internally -- transparent to the
R caller.


---

## 9. Container Inference (SPCS)

Warehouse inference (Section 7) is great for CPU-only models whose
dependencies are in the Snowflake Anaconda channel. Standard warehouses have
a ~15 GB memory limit per node, though **Snowpark-optimized warehouses** offer
significantly more memory (comparable to SPCS containers). However, warehouses
still have no GPU and are limited to Anaconda-channel packages.

**Snowpark Container Services (SPCS)** lifts these constraints. Use it when:

- The model needs **GPU acceleration** (deep learning, LLMs, embeddings)
- The model has **complex or custom dependencies** not in the Anaconda channel
- You need **low-latency / online inference** (REST endpoint, record-level scoring)
- You're running an **R model** (`rpy2` + `r-base` are not in the warehouse channel)

SPCS handles both **batch** and **online** inference. For batch, route queries
through a small warehouse to the SPCS compute pool. For online, call the
service's REST endpoint directly.

```r
# Deploy the Python model as an SPCS service
sfr_deploy_model(
  reg,
  model_name   = "SFR_DEMO_CONSUMPTION",
  version_name = "PY_V1",
  service_name = "mpg_scoring_svc",
  compute_pool = "MY_POOL",
  image_repo   = sfr_fqn(conn, "MY_IMAGE_REPO")
)

# Wait for the service to be ready
sfr_wait_for_service(reg, "mpg_scoring_svc")

# Score individual records via REST
ep <- sfr_service_endpoint(conn, "mpg_scoring_svc")
result <- sfr_predict_rest(ep, data.frame(wt = 3.0, hp = 150), conn = conn)
```

This also works for **R models** -- SPCS provides the `rpy2` + `r-base` runtime
that warehouse inference can't. See the Model Registry notebook for a full
SPCS deployment walkthrough.

| Runtime | Method | Best for |
|---------|--------|----------|
| Warehouse | `sfr_predict_sql()` / `MODEL()!predict()` | CPU models, batch ETL, Dynamic Tables, Feature Views (use Snowpark-optimized for large models) |
| SPCS (batch) | `sfr_predict()` with `service_name` | GPU models, custom deps, R models, batch scoring |
| SPCS (online) | `sfr_predict_rest()` via REST endpoint | Real-time APIs, apps, single-record scoring |
| Local R | `sfr_predict_local()` | Pre-flight testing, debugging |


---

## 10. Continuous Inference: Dynamic Tables & Inference Feature Views

For **continuous inference** on streaming data there are two approaches:

### Option A: Manual Dynamic Table

Combine `MODEL()!predict()` with a Snowflake Dynamic Table. The DT automatically
applies the model to new rows as they arrive.

```sql
CREATE OR REPLACE DYNAMIC TABLE scored_customers
    WAREHOUSE = MY_WH
    TARGET_LAG = '20 minutes'
    REFRESH_MODE = INCREMENTAL
    INITIALIZE = ON_CREATE
AS
SELECT
    customer_id,
    feature_1,
    feature_2,
    MODEL(MY_MODEL, production)!predict(feature_1, feature_2) AS prediction
FROM incoming_data;
```

### Option B: Inference Feature View (Feature Store)

Alternatively, the Feature Store can create an **inference Feature View** that
wraps `MODEL()!predict()` directly. This is a Feature View whose source query
includes the model prediction -- the FS manages the underlying Dynamic Table,
refresh schedule, and tags for you.

This is especially powerful when combined with other Feature Views: the input
features and the model predictions live in the same Feature Store lineage,
creating an end-to-end feature-engineering-to-scoring pipeline managed entirely
by the Feature Store.

See the Feature Store notebook for more on Feature Views.

### When to use which

| Approach | Best for |
|----------|----------|
| Manual DT | Simple scoring pipelines, full control over SQL/refresh |
| Inference Feature View | FS-managed lineage, combining features + predictions |

Both approaches work with warehouse-targeted models and scale to large datasets.

See: [Snowflake Docs: Continuous Model Inference with Dynamic Tables](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/warehouse#continuous-model-inference-with-dynamic-tables)


---

## 11. Cleanup


In [ ]:
%%R
# Uncomment to clean up demo objects
#
# sfr_delete_model(reg, "SFR_DEMO_CONSUMPTION")
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_DEMO_SCORING")))
# sfr_disconnect(conn)
# rcat("Cleanup complete.")


---

## Next steps

- **Model Monitoring:** See `workspace_model_monitoring.ipynb` to set up drift detection
  and performance tracking on your registered models.
- **Feature Store:** See `workspace_feature_store.ipynb` for feature engineering.
- **Model Registry:** See `workspace_model_registry.ipynb` for training and registration.


In [ ]:
_nb_elapsed = _time.time() - _nb_start
_mins, _secs = divmod(_nb_elapsed, 60)
print(f"Notebook completed: {_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total execution time: {int(_mins)}m {_secs:.1f}s")


In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.MODEL_CONSUMPTION_RESULTS AS
        SELECT 'model_consumption' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.MODEL_CONSUMPTION_RESULTS")
except Exception:
    pass
